# Análise Exploratória de Dados

- Conjunto de Dados [`Bank Customer Churn Prediction Dataset`](https://www.kaggle.com/datasets/saurabhbadole/bank-customer-churn-prediction-dataset)
- **Objetivo:** Identificar padrões e fatores associados ao churn (cancelamento) de clientes bancários.
- **Variável-alvo:** `Exited` — 1 se o cliente encerrou a conta (churned), 0 caso contrário.

## Setup

In [ ]:
# @title Carregando Bibliotecas
import pandas as pd
import numpy as np
import seaborn as sns
import itertools
import kagglehub
import os
from matplotlib import pyplot as plt
from IPython.display import Markdown, display
from scipy import stats

In [ ]:
# @title Download de dados
path = kagglehub.dataset_download("saurabhbadole/bank-customer-churn-prediction-dataset")
print("Path to dataset files:", path)

In [ ]:
# @title Importando conjunto de dados
df = pd.read_csv(os.path.join(path, "Churn_Modelling.csv"))

In [ ]:
# Definindo tema do seaborn
sns.set_theme(style="whitegrid")

## Exploração Inicial dos Dados

In [ ]:
display(Markdown("### Primeiras linhas"))
df.head()

In [ ]:
display(Markdown("### Últimas linhas"))
df.tail()

In [ ]:
display(Markdown("### Informações das variáveis"))
df.info()

- Unidades amostrais: 10.000
- Quantidade de variáveis: 14
- Taxa de churn: **~20,4%**

In [ ]:
display(Markdown("### Informações Estatísticas"))
df.describe()

In [ ]:
display(Markdown("### Valores únicos"))
df.nunique()

---

As colunas `RowNumber`, `CustomerId` e `Surname` são identificadores sem valor preditivo ou analítico e serão descartadas.

As variáveis `HasCrCard`, `IsActiveMember` e `Exited` são binárias (0/1) e serão convertidas para categoria com rótulos descritivos para facilitar a leitura dos gráficos.

---

In [ ]:
# Remove colunas de identificação sem valor analítico
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

# Converte variáveis binárias para categórico com rótulos legíveis
df['HasCrCard'] = df['HasCrCard'].map({1: 'Sim', 0: 'Não'}).astype('category')
df['IsActiveMember'] = df['IsActiveMember'].map({1: 'Ativo', 0: 'Inativo'}).astype('category')
df['Exited'] = df['Exited'].map({1: 'Churn', 0: 'Retido'}).astype('category')

df.head()

## Dicionário de Dados

In [ ]:
# @title Dicionário de dados
df_dict = pd.DataFrame([
    {"variavel": "CreditScore",      "descricao": "Score de crédito do cliente (300–850).",                          "tipo": "quantitativa", "subtipo": "discreta"},
    {"variavel": "Geography",        "descricao": "País de residência: France, Germany ou Spain.",                    "tipo": "qualitativa",  "subtipo": "nominal"},
    {"variavel": "Gender",           "descricao": "Gênero do cliente: Male ou Female.",                               "tipo": "qualitativa",  "subtipo": "nominal"},
    {"variavel": "Age",              "descricao": "Idade do cliente em anos.",                                         "tipo": "quantitativa", "subtipo": "discreta"},
    {"variavel": "Tenure",           "descricao": "Há quantos anos o cliente está no banco (0–10).",                   "tipo": "quantitativa", "subtipo": "discreta"},
    {"variavel": "Balance",          "descricao": "Saldo bancário atual do cliente.",                                  "tipo": "quantitativa", "subtipo": "contínua"},
    {"variavel": "NumOfProducts",    "descricao": "Número de produtos bancários que o cliente possui (1–4).",          "tipo": "quantitativa", "subtipo": "discreta"},
    {"variavel": "HasCrCard",        "descricao": "Indica se o cliente possui cartão de crédito (Sim/Não).",           "tipo": "qualitativa",  "subtipo": "nominal"},
    {"variavel": "IsActiveMember",   "descricao": "Indica se o cliente é membro ativo (Ativo/Inativo).",               "tipo": "qualitativa",  "subtipo": "nominal"},
    {"variavel": "EstimatedSalary",  "descricao": "Salário estimado anual do cliente.",                                "tipo": "quantitativa", "subtipo": "contínua"},
    {"variavel": "Exited",           "descricao": "Variável-alvo: Churn (encerrou conta) ou Retido.",                  "tipo": "qualitativa",  "subtipo": "nominal"},
])
display(df_dict)

## Tratamento de Valores Nulos

In [ ]:
# @title Diagnóstico dos valores nulos
display(Markdown("#### Contagem de nulos por variável"))
print(df.isnull().sum())
print(f"\nTotal de nulos: {df.isnull().sum().sum()}")

O dataset não apresenta valores nulos — nenhum tratamento de imputação é necessário. A análise pode prosseguir diretamente para a exploração estatística.

## Funções Auxiliares de Rigor Estatístico

Para variáveis contínuas assimétricas (`Balance`, `EstimatedSalary`, `CreditScore`) usa-se Spearman como medida de associação monotônica, complementando Pearson. Para comparações entre grupos (churners vs. retidos), aplica-se o teste de Mann-Whitney U com cálculo do tamanho de efeito `r` (rank-biserial).

In [ ]:
# @title Funções auxiliares

def mann_whitney_effect(data, group_col, value_col, group_a, group_b):
    """Mann-Whitney U com tamanho de efeito rank-biserial r."""
    a = data.loc[data[group_col] == group_a, value_col].dropna()
    b = data.loc[data[group_col] == group_b, value_col].dropna()
    u_stat, p_value = stats.mannwhitneyu(a, b, alternative='two-sided')
    n = len(a) * len(b)
    r = 1 - (2 * u_stat) / n
    return pd.Series({'U': u_stat, 'p': p_value, 'r': round(r, 4),
                      'mediana_a': a.median(), 'mediana_b': b.median()})


def churn_rate_por_grupo(data, group_col):
    """Taxa de churn (%) por categoria de uma variável qualitativa."""
    return (
        data.groupby(group_col, observed=True)['Exited']
        .apply(lambda s: (s == 'Churn').mean() * 100)
        .rename('churn_rate_%')
        .reset_index()
        .sort_values('churn_rate_%', ascending=False)
    )

## Perguntas e Hipóteses

> Cada pergunta recebeu um código (**Q1–Q8**) citado nos títulos dos gráficos e nas conclusões para rastreabilidade analítica.

| # | Pergunta | Hipótese |
|---|----------|----------|
| Q1 | Clientes mais velhos têm maior propensão ao churn? | Sim — clientes de maior idade costumam mudar menos de banco, mas podem ter perfil de risco diferente. |
| Q2 | A localização geográfica influencia a taxa de churn? | Sim — clientes na Alemanha podem ter taxas de churn distintas por contexto econômico e concorrência local. |
| Q3 | Clientes com maior saldo bancário fazem mais churn? | Possivelmente — saldos altos podem indicar clientes que avaliam ativamente alternativas de investimento. |
| Q4 | Membros ativos apresentam menor taxa de churn? | Sim — engajamento com o banco tende a criar retenção. |
| Q5 | O número de produtos contratados está associado ao churn? | Não-linear — clientes com 1 produto têm churn moderado; com 3–4 pode haver insatisfação. |
| Q6 | Existe diferença na taxa de churn entre homens e mulheres? | Possivelmente — diferenças de comportamento financeiro podem se manifestar no churn. |
| Q7 | O score de crédito difere entre clientes que fizeram churn e os que ficaram? | Pequena diferença — scores muito baixos podem indicar clientes em dificuldade que saem. |
| Q8 | Como o perfil combinado (Idade × Saldo × Score) caracteriza o cliente churner? | Churners tendem a ser mais velhos, com saldo mais alto e score mais baixo. |

## Análise Univariada

In [ ]:
# @title Resumo Estatístico

display(Markdown("#### Variáveis Qualitativas"))
display(df.describe(include=['object', 'category']))

display(Markdown("#### Variáveis Quantitativas"))
display(df.describe())

### Distribuição de Variáveis Quantitativas

Base para as perguntas **Q1** (Age), **Q3** (Balance), **Q7** (CreditScore).

In [ ]:
# @title Variáveis Quantitativas (univariada)

variaveis_quantitativas = df_dict.query("tipo == 'quantitativa'").variavel.tolist()

fig, axes = plt.subplots(
    figsize=(20, 8),
    ncols=len(variaveis_quantitativas),
    nrows=2,
    gridspec_kw={"height_ratios": [3, 1]}
)

for i, variavel in enumerate(variaveis_quantitativas):
    ax_hist = axes[0, i]
    ax_box  = axes[1, i]

    sns.histplot(data=df, x=variavel, ax=ax_hist, kde=True, alpha=0.8, color="steelblue")

    mediana = df[variavel].median()
    media   = df[variavel].mean()
    ax_hist.axvline(mediana, color='red',    linestyle='--', linewidth=1.2, label=f'Mediana: {mediana:.1f}')
    ax_hist.axvline(media,   color='orange', linestyle='-',  linewidth=1.2, label=f'Média: {media:.1f}')
    ax_hist.legend(fontsize=8)
    ax_hist.set_title(variavel, fontsize=11, fontweight='bold')
    ax_hist.set_xlabel('')

    sns.boxplot(data=df, x=variavel, ax=ax_box, color='steelblue', linewidth=1.2,
                flierprops=dict(marker='o', markersize=2, alpha=0.3))
    ax_box.set_xlabel(variavel, fontsize=9)

fig.suptitle("Distribuição Univariada — Variáveis Quantitativas", fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---

- **CreditScore**: distribuição aproximadamente simétrica, centrada em ~650.
- **Age**: ligeira assimetria positiva; maioria dos clientes entre 30–45 anos, com cauda à direita (clientes mais velhos).
- **Tenure**: distribuição quase uniforme entre 0 e 10 anos — o banco tem clientes de todas as «idades de relacionamento».
- **Balance**: bimodal — grande concentração de clientes com saldo zero (~3.600 casos) e uma distribuição aproximadamente normal para os demais (~100k–150k).
- **NumOfProducts**: majoritariamente 1 ou 2 produtos; poucos clientes com 3 ou 4.
- **EstimatedSalary**: distribuição uniforme entre 0 e 200k — o dataset foi gerado sinteticamente para simular diversidade salarial.

---

### Distribuição de Variáveis Qualitativas

Base para as perguntas **Q2** (Geography), **Q4** (IsActiveMember), **Q5** (NumOfProducts), **Q6** (Gender).

In [ ]:
# @title Variáveis Qualitativas (univariada)

variaveis_qualitativas = df_dict.query("tipo == 'qualitativa'").variavel.tolist()

fig, axes = plt.subplots(figsize=(16, 12), ncols=2, nrows=3)
axes = axes.flatten()

for i, variavel in enumerate(variaveis_qualitativas):
    order = df[variavel].value_counts().index

    ax = sns.countplot(
        data=df, y=variavel, ax=axes[i],
        order=order, palette='Blues_d',
        hue=variavel, legend=False, alpha=0.85
    )

    total = len(df)
    for bar in ax.patches:
        width = bar.get_width()
        if width > 0:
            ax.text(
                width + total * 0.005,
                bar.get_y() + bar.get_height() / 2,
                f'{width:.0f} ({width/total*100:.1f}%)',
                va='center', fontsize=9
            )

    ax.set_title(variavel, fontsize=12, fontweight='bold')
    ax.set_xlabel('Contagem')
    ax.set_ylabel('')
    ax.set_xlim(0, total * 0.65)

# Remove eixo sobrando se ncols*nrows > n variáveis
for j in range(len(variaveis_qualitativas), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Distribuição Univariada — Variáveis Qualitativas", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

- **Geography**: França concentra ~50% dos clientes; Alemanha e Espanha dividem os outros 50%.
- **Gender**: leve predominância masculina (~54,6% vs. 45,4%).
- **HasCrCard**: maioria (~70%) possui cartão de crédito.
- **IsActiveMember**: clientes ativos e inativos estão em proporção próxima (~51%/49%).
- **Exited**: cerca de **20,4%** dos clientes fizeram churn — desbalanceamento moderado da variável-alvo.

---

## Análise Bivariada

### Relação entre Variáveis Quantitativas

In [ ]:
# @title Comportamento par a par — variáveis quantitativas

vars_quant = ['CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary']

g = sns.pairplot(
    df[vars_quant + ['Exited']],
    hue='Exited',
    palette={'Churn': '#e74c3c', 'Retido': '#3498db'},
    diag_kind='kde',
    plot_kws={'alpha': 0.3, 's': 10},
    height=2.2
)
g.figure.suptitle("Pairplot — Variáveis Quantitativas por Status de Churn",
                  fontsize=13, fontweight='bold', y=1.01)
plt.show()

---

- **Age** é a variável quantitativa que mais separa churners de retidos: churners tendem a ser mais velhos.
- **Balance** mostra separação parcial: clientes com saldo zero raramente fazem churn; acima de ~50k a sobreposição é maior.
- `CreditScore`, `Tenure` e `EstimatedSalary` têm distribuições quase idênticas entre os dois grupos — pouca separação visual.

---

### Correlação

In [ ]:
# @title Matrizes de associação — Pearson e Spearman

cols_quant = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']
corr_pearson   = df[cols_quant].corr(method='pearson')
corr_spearman  = df[cols_quant].corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.heatmap(corr_pearson,  annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=axes[0])
axes[0].set_title('Pearson — associação linear', fontsize=13, fontweight='bold')

sns.heatmap(corr_spearman, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=axes[1])
axes[1].set_title('Spearman — associação monotônica', fontsize=13, fontweight='bold')

fig.suptitle("Matrizes de Correlação — Variáveis Quantitativas",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

- Nenhuma correlação forte entre variáveis preditoras — baixo risco de multicolinearidade.
- A correlação mais relevante é **NumOfProducts × Balance** (negativa, ~−0.3 Spearman): clientes com mais produtos tendem a ter saldo menor, sugerindo que produtos adicionais podem ser crédito/financiamento.
- As demais correlações são próximas de zero, confirmando a independência das features.

---

### Q1 — Idade × Churn

*Análise bivariada (quantitativa × qualitativa-alvo)* — distribuição etária comparada entre clientes que fizeram churn e os que foram retidos.

In [ ]:
# @title Q1 — Idade × Churn

result_q1 = mann_whitney_effect(df, 'Exited', 'Age', 'Churn', 'Retido')
print("Mann-Whitney U (Churn vs Retido) — Age:")
print(result_q1.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE
sns.kdeplot(data=df, x='Age', hue='Exited',
            palette={'Churn': '#e74c3c', 'Retido': '#3498db'},
            fill=True, alpha=0.4, ax=axes[0])
axes[0].set_title("Q1 — Distribuição de Idade por Status de Churn",
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Idade')

# Boxplot
sns.boxplot(data=df, x='Exited', y='Age',
            palette={'Churn': '#e74c3c', 'Retido': '#3498db'},
            order=['Retido', 'Churn'],
            width=0.4, linewidth=1.5,
            flierprops=dict(marker='o', markersize=2, alpha=0.2),
            ax=axes[1])
axes[1].set_title("Q1 — Boxplot de Idade por Status de Churn",
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Status')
axes[1].set_ylabel('Idade')

plt.tight_layout()
plt.show()

---

> **Resposta da Pergunta Q1.** A hipótese é **confirmada**.
>
> Clientes churners têm mediana de idade significativamente maior (~45 anos) em comparação aos retidos (~36 anos). O teste de Mann-Whitney U rejeita a hipótese nula (p < 0.001) com tamanho de efeito moderado (`r ≈ 0.32`). **Idade é o preditor univariado mais forte de churn no dataset.**

---

### Q2 — Localização Geográfica × Churn

*Análise bivariada (qualitativa × qualitativa-alvo)* — taxa de churn por país de residência.

In [ ]:
# @title Q2 — Geography × Churn

churn_geo = churn_rate_por_grupo(df, 'Geography')
print(churn_geo.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Taxa de churn por geografia
sns.barplot(
    data=churn_geo, x='Geography', y='churn_rate_%',
    palette='RdYlGn_r', ax=axes[0], order=churn_geo['Geography']
)
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.3,
                 f"{bar.get_height():.1f}%",
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title("Q2 — Taxa de Churn por País", fontsize=12, fontweight='bold')
axes[0].set_ylabel('Taxa de Churn (%)')
axes[0].set_xlabel('')
axes[0].set_ylim(0, 45)

# Contagem empilhada
cross = pd.crosstab(df['Geography'], df['Exited'], normalize='index') * 100
cross.plot(kind='bar', stacked=True,
           color={'Churn': '#e74c3c', 'Retido': '#3498db'},
           ax=axes[1], width=0.5)
axes[1].set_title("Q2 — Composição por País (%)", fontsize=12, fontweight='bold')
axes[1].set_ylabel('Proporção (%)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Status')

plt.tight_layout()
plt.show()

---

> **Resposta da Pergunta Q2.** A hipótese é **confirmada**.
>
> A **Alemanha** apresenta taxa de churn de **~32%**, quase o dobro da França (~16%) e da Espanha (~17%). A localização geográfica é um fator discriminante relevante, provavelmente refletindo diferenças regulatórias, concorrência local ou perfil demográfico distinto.

---

### Q3 — Saldo Bancário × Churn

*Análise bivariada (quantitativa × qualitativa-alvo)* — relação entre o saldo (`Balance`) e a probabilidade de churn.

In [ ]:
# @title Q3 — Balance × Churn

# Exclui saldo zero para o teste estatístico (zero é categoria comportamental distinta)
df_nonzero = df[df['Balance'] > 0].copy()
result_q3 = mann_whitney_effect(df_nonzero, 'Exited', 'Balance', 'Churn', 'Retido')
print("Mann-Whitney U (saldo > 0, Churn vs Retido) — Balance:")
print(result_q3.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE apenas para saldo > 0
sns.kdeplot(data=df_nonzero, x='Balance', hue='Exited',
            palette={'Churn': '#e74c3c', 'Retido': '#3498db'},
            fill=True, alpha=0.4, ax=axes[0])
axes[0].set_title("Q3 — Distribuição de Saldo (> 0) por Status",
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Saldo (R$)')

# Proporção de saldo zero vs. não-zero por churn
df['zero_balance'] = (df['Balance'] == 0).map({True: 'Saldo Zero', False: 'Saldo > 0'})
cross_bal = pd.crosstab(df['zero_balance'], df['Exited'], normalize='index') * 100
cross_bal.plot(kind='bar', stacked=True,
               color={'Churn': '#e74c3c', 'Retido': '#3498db'},
               ax=axes[1], width=0.4)
axes[1].set_title("Q3 — Churn por Categoria de Saldo",
                  fontsize=12, fontweight='bold')
axes[1].set_ylabel('Proporção (%)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Status')

df.drop(columns=['zero_balance'], inplace=True)
plt.tight_layout()
plt.show()

---

> **Resposta da Pergunta Q3.** A hipótese é **parcialmente confirmada**.
>
> Clientes com **saldo zero (~36% da base)** têm taxa de churn muito baixa (~13%), enquanto clientes com saldo positivo têm churn de ~25%. Entre os que possuem saldo, churners têm mediana ligeiramente maior — o teste de Mann-Whitney U é significativo (p < 0.001), mas o tamanho de efeito é pequeno (`r ≈ 0.09`). A variável `Balance` sozinha não discrimina bem o churn, mas a dicotomia saldo-zero vs. saldo-positivo é relevante.

---

### Q4 — Membro Ativo × Churn

*Análise bivariada (qualitativa × qualitativa-alvo)* — comparação da taxa de churn entre membros ativos e inativos.

In [ ]:
# @title Q4 — IsActiveMember × Churn

churn_active = churn_rate_por_grupo(df, 'IsActiveMember')
print(churn_active.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.barplot(
    data=churn_active, x='IsActiveMember', y='churn_rate_%',
    palette={'Inativo': '#e74c3c', 'Ativo': '#2ecc71'},
    hue='IsActiveMember', legend=False,
    order=['Inativo', 'Ativo'], ax=axes[0]
)
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.3,
                 f"{bar.get_height():.1f}%",
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_title("Q4 — Taxa de Churn por Status de Atividade",
                  fontsize=12, fontweight='bold')
axes[0].set_ylabel('Taxa de Churn (%)')
axes[0].set_xlabel('')
axes[0].set_ylim(0, 35)

cross_act = pd.crosstab(df['IsActiveMember'], df['Exited'], normalize='index') * 100
cross_act.plot(kind='bar', stacked=True,
               color={'Churn': '#e74c3c', 'Retido': '#3498db'},
               ax=axes[1], width=0.4)
axes[1].set_title("Q4 — Composição por Status de Atividade (%)",
                  fontsize=12, fontweight='bold')
axes[1].set_ylabel('Proporção (%)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Status')

plt.tight_layout()
plt.show()

---

> **Resposta da Pergunta Q4.** A hipótese é **confirmada**.
>
> Membros **inativos** têm taxa de churn de **~27%**, enquanto membros **ativos** apresentam apenas **~14%**. O engajamento com o banco é um fator de retenção relevante — clientes que utilizam ativamente os produtos têm quase metade da probabilidade de churn comparados aos inativos.

---

### Q5 — Número de Produtos × Churn

*Análise bivariada (quantitativa discreta × qualitativa-alvo)* — taxa de churn por número de produtos contratados.

In [ ]:
# @title Q5 — NumOfProducts × Churn

churn_prod = churn_rate_por_grupo(df, 'NumOfProducts')
print(churn_prod.sort_values('NumOfProducts').to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order_prod = sorted(df['NumOfProducts'].unique())
churn_plot = churn_prod.set_index('NumOfProducts').reindex(order_prod).reset_index()

sns.barplot(
    data=churn_plot, x='NumOfProducts', y='churn_rate_%',
    palette='RdYlGn_r', ax=axes[0]
)
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5,
                 f"{bar.get_height():.1f}%",
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title("Q5 — Taxa de Churn por Nº de Produtos",
                  fontsize=12, fontweight='bold')
axes[0].set_ylabel('Taxa de Churn (%)')
axes[0].set_xlabel('Número de Produtos')
axes[0].set_ylim(0, 110)

cross_prod = pd.crosstab(df['NumOfProducts'], df['Exited'], normalize='index') * 100
cross_prod.plot(kind='bar', stacked=True,
                color={'Churn': '#e74c3c', 'Retido': '#3498db'},
                ax=axes[1], width=0.5)
axes[1].set_title("Q5 — Composição por Nº de Produtos (%)",
                  fontsize=12, fontweight='bold')
axes[1].set_ylabel('Proporção (%)')
axes[1].set_xlabel('Número de Produtos')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Status')

plt.tight_layout()
plt.show()

---

> **Resposta da Pergunta Q5.** A hipótese é **confirmada com padrão não-linear**.
>
> - **1 produto**: churn de ~28% — clientes com baixo engajamento com produtos.
> - **2 produtos**: churn de apenas ~8% — ponto ótimo de retenção.
> - **3 produtos**: churn de ~83% — inversão dramática, possivelmente clientes insatisfeitos com o portfólio.
> - **4 produtos**: churn de ~100% — todos os clientes com 4 produtos encerraram a conta (n pequeno, ~60 clientes).
>
> **2 produtos é o ponto de máxima retenção**; ter 3 ou 4 produtos é um forte sinal de alerta para churn.

---

### Q6 — Gênero × Churn

*Análise bivariada (qualitativa × qualitativa-alvo)* — diferença na taxa de churn entre gêneros.

In [ ]:
# @title Q6 — Gender × Churn

churn_gen = churn_rate_por_grupo(df, 'Gender')
print(churn_gen.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.barplot(
    data=churn_gen, x='Gender', y='churn_rate_%',
    palette={'Female': '#e91e8c', 'Male': '#1e88e5'},
    hue='Gender', legend=False,
    order=['Female', 'Male'], ax=axes[0]
)
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.3,
                 f"{bar.get_height():.1f}%",
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_title("Q6 — Taxa de Churn por Gênero",
                  fontsize=12, fontweight='bold')
axes[0].set_ylabel('Taxa de Churn (%)')
axes[0].set_xlabel('')
axes[0].set_ylim(0, 32)

cross_gen = pd.crosstab(df['Gender'], df['Exited'], normalize='index') * 100
cross_gen.plot(kind='bar', stacked=True,
               color={'Churn': '#e74c3c', 'Retido': '#3498db'},
               ax=axes[1], width=0.35)
axes[1].set_title("Q6 — Composição por Gênero (%)",
                  fontsize=12, fontweight='bold')
axes[1].set_ylabel('Proporção (%)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Status')

plt.tight_layout()
plt.show()

---

> **Resposta da Pergunta Q6.** A hipótese é **confirmada**.
>
> Clientes do gênero **feminino** apresentam taxa de churn de **~25%**, versus **~16%** para o gênero masculino. A diferença é estatisticamente relevante — mulheres têm aproximadamente 56% mais chance de churn do que homens neste dataset. Isso pode refletir diferenças comportamentais, de produto ou de atendimento que merecem investigação qualitativa.

---

### Q7 — Score de Crédito × Churn

*Análise bivariada (quantitativa × qualitativa-alvo)* — distribuição de CreditScore comparada entre grupos.

In [ ]:
# @title Q7 — CreditScore × Churn

result_q7 = mann_whitney_effect(df, 'Exited', 'CreditScore', 'Churn', 'Retido')
print("Mann-Whitney U (Churn vs Retido) — CreditScore:")
print(result_q7.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.kdeplot(data=df, x='CreditScore', hue='Exited',
            palette={'Churn': '#e74c3c', 'Retido': '#3498db'},
            fill=True, alpha=0.4, ax=axes[0])
axes[0].set_title("Q7 — Distribuição de CreditScore por Status",
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Score de Crédito')

sns.violinplot(data=df, x='Exited', y='CreditScore',
               palette={'Churn': '#e74c3c', 'Retido': '#3498db'},
               order=['Retido', 'Churn'],
               inner='quartile', linewidth=1.2, ax=axes[1])
axes[1].set_title("Q7 — Violin Plot de CreditScore por Status",
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Status')
axes[1].set_ylabel('Score de Crédito')

plt.tight_layout()
plt.show()

---

> **Resposta da Pergunta Q7.** A hipótese é **parcialmente confirmada**.
>
> Existe diferença estatisticamente significativa (p < 0.05), mas o tamanho de efeito é **muito pequeno** (`r ≈ 0.03`). As distribuições de score são quase idênticas entre churners e retidos. `CreditScore` sozinho não discrimina o churn — o score de crédito neste banco cobre uma faixa semelhante independentemente do status de saída.

---

## Análise Multivariada

### Q8 — Perfil Multivariado do Churner (Idade × Saldo × Score × Geografia)

In [ ]:
# @title Q8 — Dispersão Idade × Saldo colorida por Churn e segmentada por País

df_plot = df[df['Balance'] > 0].copy()

g = sns.FacetGrid(
    df_plot, col='Geography', hue='Exited',
    palette={'Churn': '#e74c3c', 'Retido': '#3498db'},
    height=4.5, aspect=1.1, col_wrap=3
)
g.map_dataframe(
    sns.scatterplot,
    x='Age', y='Balance',
    alpha=0.35, s=18, edgecolor='none'
)
g.add_legend(title='Status', bbox_to_anchor=(1.02, 0.5))
g.set_axis_labels('Idade', 'Saldo (Balance)')
g.set_titles(col_template="{col_name}")
g.figure.suptitle(
    "Q8 — Idade × Saldo por País e Status de Churn (saldo > 0)",
    fontsize=13, fontweight='bold', y=1.02
)
plt.show()

In [ ]:
# @title Q8 — Mapa de Calor: Churn por Faixa Etária × Número de Produtos

df['faixa_etaria'] = pd.cut(
    df['Age'],
    bins=[17, 30, 40, 50, 60, 92],
    labels=['18–30', '31–40', '41–50', '51–60', '61+']
)

pivot_q8 = (
    df.groupby(['faixa_etaria', 'NumOfProducts'], observed=True)['Exited']
    .apply(lambda s: (s == 'Churn').mean() * 100)
    .unstack('NumOfProducts')
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    pivot_q8, annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.5, cbar_kws={'label': 'Churn Rate (%)'},
    ax=ax
)
ax.set_title("Q8 — Taxa de Churn (%) por Faixa Etária × Nº de Produtos",
             fontsize=13, fontweight='bold')
ax.set_xlabel('Número de Produtos')
ax.set_ylabel('Faixa Etária')
plt.tight_layout()
plt.show()

---

> **Resposta da Pergunta Q8.** A hipótese é **confirmada e ampliada**.
>
> - A combinação **mais crítica** é: clientes com **41–60 anos** e **3–4 produtos** têm taxas de churn entre 80–100%, independentemente do país.
> - Na Alemanha, o padrão de churn por idade é mais distribuído — clientes de todas as faixas etárias apresentam taxas elevadas, o que reforça o achado de Q2.
> - **2 produtos continua sendo o patamar de maior retenção** em todas as faixas etárias.
> - A interação Idade × NumOfProducts é o vetor mais informativo para identificar clientes em risco de churn.

---

## Análise dos Resultados

### Síntese Geral

Esta EDA investigou 10.000 registros de clientes de um banco cobrindo 3 países (França, Alemanha e Espanha), com 11 variáveis analíticas após remoção dos identificadores. A taxa de churn da base é de **20,4%** — desbalanceamento moderado a ser levado em conta em eventuais modelos preditivos.

Os fatores mais discriminantes identificados na análise exploratória foram:
1. **Idade** — churners são significativamente mais velhos (mediana ~45 vs. ~36 anos).
2. **Número de produtos** — churn quase nulo com 2 produtos; quase total com 3–4 produtos.
3. **Status de atividade** — membros inativos fazem churn quase 2× mais do que membros ativos.
4. **País (Alemanha)** — taxa de churn ~32%, cerca do dobro dos demais países.
5. **Gênero feminino** — taxa de churn ~25% vs. ~16% para o gênero masculino.

Variáveis com poder discriminante fraco: `CreditScore`, `Tenure`, `EstimatedSalary`, `HasCrCard`.

# Resultados

## Tabela de Perguntas, Hipóteses e Resultados

| # | Pergunta | Hipótese | Resultado | Achado-chave | Confirmada? |
|---|----------|----------|-----------|--------------|-------------|
| Q1 | Clientes mais velhos têm maior churn? | Sim | Mediana: 45 (churn) vs. 36 (retido); Mann-Whitney r=0.32 | **Idade é o preditor univariado mais forte** | ✅ |
| Q2 | A localização geográfica influencia o churn? | Sim | Alemanha: ~32%; França/Espanha: ~16-17% | **Alemanha tem 2× mais churn** | ✅ |
| Q3 | Clientes com maior saldo fazem mais churn? | Parcialmente | Saldo zero → baixo churn; saldo positivo → efeito pequeno (r=0.09) | **Dicotomia saldo-zero é mais relevante que o valor absoluto** | ⚠️ Parcial |
| Q4 | Membros ativos têm menor churn? | Sim | Ativos: ~14%; Inativos: ~27% | **Engajamento é fator de retenção relevante** | ✅ |
| Q5 | Nº de produtos está associado ao churn? | Sim (não-linear) | 1 prod: 28%; 2 prod: 8%; 3 prod: 83%; 4 prod: ~100% | **2 produtos = ponto ótimo de retenção** | ✅ |
| Q6 | Há diferença de churn por gênero? | Sim | Female: ~25%; Male: ~16% | **Clientes femininas têm 56% mais churn** | ✅ |
| Q7 | CreditScore difere entre churners e retidos? | Pequena diferença | Diferença significativa mas tamanho de efeito mínimo (r=0.03) | **CreditScore não discrimina o churn** | ⚠️ Parcial |
| Q8 | O perfil multivariado caracteriza o churner? | Sim | Churners: 41–60 anos, 3–4 produtos, Alemanha | **Interação Idade × Produtos é o vetor mais informativo** | ✅ |